<a href="https://colab.research.google.com/github/UdaraChamidu/Eye-Disease-Classification-With-Integrated-Chatbot/blob/main/Fine_Tune_bioGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install Required Libraries

In [ ]:
pip install transformers datasets accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 664.8/664.8 MB 149.4 MB/s eta 0:00:01

# Step 2: Load Dataset (.jsonl)

In [ ]:
from datasets import Dataset
import json

# Load your JSONL file
file_path = "/content/cleaned_dataset.jsonl"

# Step 1: Read the lines into a list of dicts
with open(file_path, "r", encoding="utf-8") as f:
    data_lines = [json.loads(line) for line in f]

# Step 2: Convert to Hugging Face Dataset
dataset = Dataset.from_list(data_lines)

# Step 3: Format the dataset correctly using the right keys
def format_example(example):
    return {
        "text": f"### Question:\n{example['prompt']}\n\n### Answer:\n{example['completion']}"
    }

# Step 4: Apply formatting
dataset = dataset.map(format_example)


In [ ]:
print(dataset[0]["text"])

# Step 3: Load the Pretrained BioGPT Model

In [ ]:
!pip install -q sacremoses


In [ ]:
from transformers import BioGptTokenizer, BioGptForCausalLM

tokenizer = BioGptTokenizer.from_pretrained("microsoft/biogpt")
model = BioGptForCausalLM.from_pretrained("microsoft/biogpt")
# model_name = "microsoft/biogpt"

# Step 4: Tokenize the Data

In [ ]:
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=512)

tokenized_data = dataset.map(tokenize, batched=True, remove_columns=['prompt', 'completion', 'text'])


# Step 5: Define Training Arguments

In [ ]:
!pip uninstall -y transformers
!pip install transformers --upgrade


In [ ]:
import transformers
print(transformers.__version__)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./biogpt-finetuned",
    #evaluation_strategy="no",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)


# Step 6: Initialize Trainer and Train

In [ ]:
from transformers import BioGptTokenizer

tokenizer = BioGptTokenizer.from_pretrained("microsoft/biogpt")


In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()


# Step 7: Save and Use Your Fine-Tuned Model

In [ ]:
trainer.save_model("./biogpt-finetuned")
tokenizer.save_pretrained("./biogpt-finetuned")


In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="./biogpt-finetuned", tokenizer="./biogpt-finetuned")
pipe("### Question:\nWhat is cataract?\n\n### Answer:\n", max_new_tokens=100)
